# Install required packages

In [5]:
!pip install streamlit
!pip install mysql-connector-python

# Importing Libraries

In [7]:
# Import necessary Python libraries

import requests                        # For making API requests
from datetime import datetime          # For working with dates
import mysql.connector                 # For connecting to MySQL databases
import pandas as pd                    # For handling and manipulating dataframes
import streamlit as st                 # For building the web app

Get API key

In [9]:
#NASA API key
API_key = "DytJfcrM7uHY6TxR3Jj48G3d3egHJmgI51chljPF"

# Construct the NASA NEO Feed URL
# Fetching data for date range: 2024-01-01 to 2024-01-08
url = f"https://api.nasa.gov/neo/rest/v1/feed?start_date=2024-1-1&end_date=2024-1-8&api_key={API_key}"

# Make a GET request to NASA API
response = requests.get(url)

# Check HTTP Status Code
print("Status Code:", response.status_code)

# Verifying if the API request was successful
if response.status_code == 200:
    print("API key is valid ✅")

     # Parse the JSON data from the response
    data = response.json()

    # Print the keys of the main JSON dictionary (for understanding the structure)
    print("sample data", data.keys())    # Should show: 'keys'

else:

    # If the response is not 200, print an error message
    print("API key is invalid ❌", response.json())



Status Code: 200
API key is valid ✅
sample data dict_keys(['links', 'element_count', 'near_earth_objects'])


# --- Exploring the structure of the received JSON ---

In [11]:
data['near_earth_objects'].items() # Viewing all (date, asteroids_list)

dict_items([('2024-01-02', [{'links': {'self': 'http://api.nasa.gov/neo/rest/v1/neo/2415949?api_key=DytJfcrM7uHY6TxR3Jj48G3d3egHJmgI51chljPF'}, 'id': '2415949', 'neo_reference_id': '2415949', 'name': '415949 (2001 XY10)', 'nasa_jpl_url': 'https://ssd.jpl.nasa.gov/tools/sbdb_lookup.html#/?sstr=2415949', 'absolute_magnitude_h': 19.37, 'estimated_diameter': {'kilometers': {'estimated_diameter_min': 0.3552670883, 'estimated_diameter_max': 0.7944013596}, 'meters': {'estimated_diameter_min': 355.267088298, 'estimated_diameter_max': 794.4013596028}, 'miles': {'estimated_diameter_min': 0.2207526659, 'estimated_diameter_max': 0.4936179672}, 'feet': {'estimated_diameter_min': 1165.5744739718, 'estimated_diameter_max': 2606.3037566394}}, 'is_potentially_hazardous_asteroid': False, 'close_approach_data': [{'close_approach_date': '2024-01-02', 'close_approach_date_full': '2024-Jan-02 13:14', 'epoch_date_close_approach': 1704201240000, 'relative_velocity': {'kilometers_per_second': '15.8905264223', 

In [13]:
# Show only the available dates (keys) in the dataset
data['near_earth_objects'].keys()

dict_keys(['2024-01-02', '2024-01-01', '2024-01-04', '2024-01-03', '2024-01-06', '2024-01-05', '2024-01-08', '2024-01-07'])

In [15]:
# View details of the first asteroid object from a specific date (e.g., 2024-01-01)
data['near_earth_objects']['2024-01-01'][0]

{'links': {'self': 'http://api.nasa.gov/neo/rest/v1/neo/3724393?api_key=DytJfcrM7uHY6TxR3Jj48G3d3egHJmgI51chljPF'},
 'id': '3724393',
 'neo_reference_id': '3724393',
 'name': '(2015 OD22)',
 'nasa_jpl_url': 'https://ssd.jpl.nasa.gov/tools/sbdb_lookup.html#/?sstr=3724393',
 'absolute_magnitude_h': 21.21,
 'estimated_diameter': {'kilometers': {'estimated_diameter_min': 0.152249185,
   'estimated_diameter_max': 0.3404395273},
  'meters': {'estimated_diameter_min': 152.249185036,
   'estimated_diameter_max': 340.4395272595},
  'miles': {'estimated_diameter_min': 0.0946032284,
   'estimated_diameter_max': 0.2115392495},
  'feet': {'estimated_diameter_min': 499.5052162336,
   'estimated_diameter_max': 1116.9276186141}},
 'is_potentially_hazardous_asteroid': False,
 'close_approach_data': [{'close_approach_date': '2024-01-01',
   'close_approach_date_full': '2024-Jan-01 11:58',
   'epoch_date_close_approach': 1704110280000,
   'relative_velocity': {'kilometers_per_second': '25.3931645932',
  

# --- Collecting Data ---

In [17]:
# Create an empty list to store all asteroid records
asteroids_data = []

# Set a target to collect only up to 10,000 records
target = 10000

# Start fetching data until either there's no next URL or we reach 10,000 records
while len(asteroids_data) < target:

    # Make a GET request to the current url
    response = requests.get(url)

    # Parse the JSON response
    data = response.json()

    # Extract the dictionary of near-Earth objects grouped by date
    neo_data = data['near_earth_objects']

    # Loop through each date and its corresponding list of asteroids
    for date, asteroids in neo_data.items():
        for asteroid in asteroids:

            # Prepare and clean asteroid data with correct formats
            asteroid_data = {
                'id': int(asteroid['id']),  # Unique asteroid ID
                'name': asteroid['name'],  # Asteroid name
                'absolute_magnitude_h': float(asteroid['absolute_magnitude_h']),  # Brightness measure
                'estimated_diameter_min_km': float(asteroid['estimated_diameter']['kilometers']['estimated_diameter_min']),  # Minimum diameter (km)
                'estimated_diameter_max_km': float(asteroid['estimated_diameter']['kilometers']['estimated_diameter_max']),  # Maximum diameter (km)
                'is_potentially_hazardous': asteroid['is_potentially_hazardous_asteroid'],  # Hazardous status (True/False)
                'close_approach_date': datetime.strptime(asteroid['close_approach_data'][0]['close_approach_date'], "%Y-%m-%d"),  # Date of closest approach
                'relative_velocity_kmh': float(asteroid['close_approach_data'][0]['relative_velocity']['kilometers_per_hour']),  # Approach speed (km/h)
                'astronomical': float(asteroid['close_approach_data'][0]['miss_distance']['astronomical']),  # Distance in Astronomical Units (AU)
                'miss_distance_km': float(asteroid['close_approach_data'][0]['miss_distance']['kilometers']),  # Distance missed in kilometers
                'miss_distance_lunar': float(asteroid['close_approach_data'][0]['miss_distance']['lunar']),  # Distance in Lunar Distances (LD)
                'orbiting_body': asteroid['close_approach_data'][0]['orbiting_body']  # Orbiting body (should be "Earth")
            }

            # Add the asteroid's data dictionary to the main list
            asteroids_data.append(asteroid_data)

            # Check if we have already collected 10,000 records
            if len(asteroids_data) >= target:
                break  # If yes, stop processing further asteroids

        # Double-check at the date-level loop if limit is reached
        if len(asteroids_data) >= target:
            break

# Update url to the 'next' link for the next 7-day batch
url = data['links'].get('next')

#Check number of records
print("Successfully collected data = ", len(asteroids_data)) #This will confirm if you actually collected 10,000 rows.


Successfully collected data =  10000


# --- CLEANING DATA ---

In [19]:
# View first asteroiod
asteroids_data[0]

{'id': 2415949,
 'name': '415949 (2001 XY10)',
 'absolute_magnitude_h': 19.37,
 'estimated_diameter_min_km': 0.3552670883,
 'estimated_diameter_max_km': 0.7944013596,
 'is_potentially_hazardous': False,
 'close_approach_date': datetime.datetime(2024, 1, 2, 0, 0),
 'relative_velocity_kmh': 57205.8951204341,
 'astronomical': 0.3372535274,
 'miss_distance_km': 50452409.349026635,
 'miss_distance_lunar': 131.1916221586,
 'orbiting_body': 'Earth'}

In [21]:
#View first few asteroids

# Print first 5 records This helps us quickly check what our collected data looks like.
for asteroid in asteroids_data[:5]:
    print(asteroid)
    print("-" * 50)



{'id': 2415949, 'name': '415949 (2001 XY10)', 'absolute_magnitude_h': 19.37, 'estimated_diameter_min_km': 0.3552670883, 'estimated_diameter_max_km': 0.7944013596, 'is_potentially_hazardous': False, 'close_approach_date': datetime.datetime(2024, 1, 2, 0, 0), 'relative_velocity_kmh': 57205.8951204341, 'astronomical': 0.3372535274, 'miss_distance_km': 50452409.349026635, 'miss_distance_lunar': 131.1916221586, 'orbiting_body': 'Earth'}
--------------------------------------------------
{'id': 3160747, 'name': '(2003 SR84)', 'absolute_magnitude_h': 26.0, 'estimated_diameter_min_km': 0.0167708462, 'estimated_diameter_max_km': 0.0375007522, 'is_potentially_hazardous': False, 'close_approach_date': datetime.datetime(2024, 1, 2, 0, 0), 'relative_velocity_kmh': 38589.054833182, 'astronomical': 0.1323425924, 'miss_distance_km': 19798169.933318187, 'miss_distance_lunar': 51.4812684436, 'orbiting_body': 'Earth'}
--------------------------------------------------
{'id': 3309828, 'name': '(2005 YQ96)

In [23]:
# Load into pandas for better view

df = pd.DataFrame(asteroids_data)

# Display top 10 rows
print(df.head(10))

        id                name  absolute_magnitude_h  \
0  2415949  415949 (2001 XY10)                 19.37   
1  3160747         (2003 SR84)                 26.00   
2  3309828         (2005 YQ96)                 20.62   
3  3457842         (2009 HC21)                 22.10   
4  3553062         (2010 XA11)                 26.10   
5  3591616         (2011 YP10)                 23.94   
6  3608936         (2012 SD22)                 20.05   
7  3795154          (2017 YD8)                 21.87   
8  3842680          (2019 KK5)                 22.79   
9  2613286  613286 (2005 YQ96)                 20.63   

   estimated_diameter_min_km  estimated_diameter_max_km  \
0                   0.355267                   0.794401   
1                   0.016771                   0.037501   
2                   0.199781                   0.446725   
3                   0.101054                   0.225964   
4                   0.016016                   0.035813   
5                   0.043307 

In [25]:
# Check missing/null value

df.isnull().sum()


id                           0
name                         0
absolute_magnitude_h         0
estimated_diameter_min_km    0
estimated_diameter_max_km    0
is_potentially_hazardous     0
close_approach_date          0
relative_velocity_kmh        0
astronomical                 0
miss_distance_km             0
miss_distance_lunar          0
orbiting_body                0
dtype: int64

In [37]:
# create a csv file

df.to_csv('asteroids.csv', index=False)

# --- CREATING MYSQL CONNECTION ---

In [85]:
#Mysql Database connection

db_connection = mysql.connector.connect(
    host='localhost',
    user='root',
    password='998800',
    database='nasa_db'
)

if db_connection.is_connected:
    print("DATABASE SUCCESSFULLY CONNECTED ✅")

else:
    print("ERROR: DATABASE not connected ❌")



DATABASE SUCCESSFULLY CONNECTED ✅


In [39]:
# creating tables asteroids and close approach

cursor = db_connection.cursor()

cursor.execute('''
        CREATE TABLE IF NOT EXISTS asteroids (
            id VARCHAR(50),
            name VARCHAR(255),
            absolute_magnitude_h FLOAT,
            estimated_diameter_min_km FLOAT,
            estimated_diameter_max_km FLOAT,
            is_potentially_hazardous_asteroid BOOLEAN
        );
    ''')
db_connection.commit()

In [45]:
# insert values to tables

cursor = db_connection.cursor()

# Insert Query ( asteroids)
insert_query = '''
INSERT INTO asteroids (id, name, absolute_magnitude_h, estimated_diameter_min_km, estimated_diameter_max_km, is_potentially_hazardous_asteroid)
VALUES (%s, %s, %s, %s, %s, %s)
'''

# Iterate over the DataFrame and insert each record
for index, row in df.iterrows():
    cursor.execute(insert_query, (
        row['id'],
        row['name'],
        row['absolute_magnitude_h'],       
        row['estimated_diameter_min_km'],  
        row['estimated_diameter_max_km'],   
        row['is_potentially_hazardous']
    ))

# Commit the changes
db_connection.commit()


In [87]:
# creating tables asteroids and close approach

cursor = db_connection.cursor()
cursor.execute('''
        CREATE TABLE IF NOT EXISTS close_approach (
            neo_reference_id VARCHAR(50),
            close_approach_date DATE,
            relative_velocity_kmph FLOAT,
            astronomical FLOAT,
            miss_distance_km FLOAT,
            miss_distance_lunar FLOAT,
            orbiting_body VARCHAR(50)
        );
    ''')
db_connection.commit()

In [91]:
# insert values to tables

cursor = db_connection.cursor()

# Insert data into the close_aproach table
insert_query = '''
INSERT INTO close_approach (neo_reference_id, close_approach_date, relative_velocity_kmph, astronomical, miss_distance_km, miss_distance_lunar, orbiting_body)
VALUES (%s, %s, %s, %s, %s, %s, %s)
'''

# Iterate over the DataFrame and insert each record
for index, row in df.iterrows():
    cursor.execute(insert_query, (
        row['id'],
        row['close_approach_date'],
        row['relative_velocity_kmh'],
        row['astronomical'],
        row['miss_distance_km'],
        row['miss_distance_lunar'],
        row['orbiting_body']
    ))

# Commit the changes to the database
db_connection.commit()

In [95]:
cursor.execute("Select * from asteroids")
results = cursor.fetchall()

pd.DataFrame(results,columns = [i[0]for i in cursor.description])

,id,name,absolute_magnitude_h,estimated_diameter_min_km,estimated_diameter_max_km,is_potentially_hazardous_asteroid
0,2415949,415949 (2001 XY10),19.37,0.355267,0.794401,0
1,3160747,(2003 SR84),26.00,0.016771,0.037501,0
2,3309828,(2005 YQ96),20.62,0.199781,0.446725,1
3,3457842,(2009 HC21),22.10,0.101054,0.225964,0
4,3553062,(2010 XA11),26.10,0.016016,0.035813,0
...,...,...,...,...,...,...
9995,54448604,(2024 MM),24.50,0.033462,0.074824,0
9996,2199003,199003 (2005 WJ56),18.16,0.620233,1.386880,1
9997,2434188,434188 (2003 AD23),19.09,0.404162,0.903733,1
9998,3398095,(2007 YZ),19.53,0.330031,0.737972,0


In [97]:
cursor.execute("Select * from close_approach")
results = cursor.fetchall()

import pandas as pd

pd.DataFrame(results,columns = [i[0]for i in cursor.description])

,neo_reference_id,close_approach_date,relative_velocity_kmph,astronomical,miss_distance_km,miss_distance_lunar,orbiting_body
0,2415949,2024-01-02,57205.9,0.337254,50452400.0,131.1920,Earth
1,3160747,2024-01-02,38589.1,0.132343,19798200.0,51.4813,Earth
2,3309828,2024-01-02,56413.0,0.167013,24984700.0,64.9679,Earth
3,3457842,2024-01-02,21891.1,0.492051,73609800.0,191.4080,Earth
4,3553062,2024-01-02,31469.0,0.235802,35275500.0,91.7271,Earth
...,...,...,...,...,...,...,...
9995,54448604,2024-01-08,27845.1,0.198545,29701900.0,77.2339,Earth
9996,2199003,2024-01-07,55047.0,0.199535,29849900.0,77.6189,Earth
9997,2434188,2024-01-07,121814.0,0.204314,30564900.0,79.4781,Earth
9998,3398095,2024-01-07,63326.1,0.191767,28687900.0,74.5972,Earth


# --- QUERIES ---

In [101]:
# list top 10 fastest asteroids

cursor.execute("""SELECT name,relative_velocity_kmph FROM close_approach
                join asteroids
               on asteroids.id = close_approach.neo_reference_id
               order by relative_velocity_kmph desc
               limit 10""")
result = cursor.fetchall()
data = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
data

,name,relative_velocity_kmph
0,(2013 NT11),136268.0
1,(2013 NT11),136268.0
2,(2013 NT11),136268.0
3,(2013 NT11),136268.0
4,(2013 NT11),136268.0
5,(2013 NT11),136268.0
6,(2013 NT11),136268.0
7,(2013 NT11),136268.0
8,(2013 NT11),136268.0
9,(2013 NT11),136268.0


In [103]:
cursor.execute("""select close_approach_date, avg(relative_velocity_kmph) from close_approach
                  group by close_approach_date
                  order by avg(relative_velocity_kmph) desc""" )

result = cursor.fetchall()
data = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
data



,close_approach_date,avg(relative_velocity_kmph)
0,2024-01-07,51301.760874
1,2024-01-05,49251.419108
2,2024-01-06,48055.191817
3,2024-01-01,47485.112305
4,2024-01-03,43949.719889
5,2024-01-04,43755.678516
6,2024-01-02,42740.480291
7,2024-01-08,41904.075709
